# 02 - Data Cleaning & Analytical Preprocessing

**Project**: Retail Customer, Product & Promotion Intelligence  
**Dataset**: dunnhumby – The Complete Journey  

### Objectives:
1. Standardize types, clean whitespace, and handle anomalies across all 7 raw datasets.
2. Preserve the original transaction-level grain (2,595,732 rows) without fabricating or silently discarding data.
3. Transform negative discount representations into intuitive business metrics (retail discounts, coupon discounts, out-of-pocket customer spend).
4. Remove exact duplicate mappings in `coupon.csv` (5,164 duplicates).
5. Build curated analytical aggregation tables:
   - **Basket Level** (`baskets.csv`): 276,484 shopping trips.
   - **Household Level** (`household_summary.csv`): 2,500 households with purchasing cadence and demographic tags.
6. Export cleaned tables to `data/processed/` for downstream Python, SQL, and Power BI analysis.

## 1. Environment Setup & Data Ingestion
Loading dependencies and raw datasets.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path for src modules
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_cleaning.cleaners import (
    clean_transactions,
    clean_products,
    clean_demographics,
    clean_campaigns,
    clean_coupons,
    clean_coupon_redemptions,
    build_basket_summary,
    build_household_summary
)

print(f"Project root directory: {ROOT_DIR}")

Project root directory: /Users/anshikagupta/Developer/retail-customer-product-and-promotion-intelligence


In [2]:
# Load raw datasets
raw_dir = ROOT_DIR / "data" / "raw"

tx_raw = pd.read_csv(raw_dir / "transaction_data.csv")
prod_raw = pd.read_csv(raw_dir / "product.csv")
demo_raw = pd.read_csv(raw_dir / "hh_demographic.csv")
camp_desc_raw = pd.read_csv(raw_dir / "campaign_desc.csv")
camp_table_raw = pd.read_csv(raw_dir / "campaign_table.csv")
coupon_raw = pd.read_csv(raw_dir / "coupon.csv")
redempt_raw = pd.read_csv(raw_dir / "coupon_redempt.csv")

print("Raw datasets loaded successfully:")
print(f" - Transactions: {tx_raw.shape[0]:,} rows, {tx_raw.shape[1]} cols")
print(f" - Products:     {prod_raw.shape[0]:,} rows, {prod_raw.shape[1]} cols")
print(f" - Demographics: {demo_raw.shape[0]:,} rows, {demo_raw.shape[1]} cols")
print(f" - Campaigns:    {camp_desc_raw.shape[0]:,} campaigns")
print(f" - Camp Target:  {camp_table_raw.shape[0]:,} household targeting rows")
print(f" - Coupons:      {coupon_raw.shape[0]:,} coupon mappings")
print(f" - Redemptions:  {redempt_raw.shape[0]:,} redemption rows")

Raw datasets loaded successfully:
 - Transactions: 2,595,732 rows, 12 cols
 - Products:     92,353 rows, 7 cols
 - Demographics: 801 rows, 8 cols
 - Campaigns:    30 campaigns
 - Camp Target:  7,208 household targeting rows
 - Coupons:      124,548 coupon mappings
 - Redemptions:  2,318 redemption rows


## 2. Transaction Data Cleaning & Business Metric Derivation

**Cleaning Rules & Audit**:
- In the dunnhumby dataset, `SALES_VALUE` is the dollar amount received by the retailer (net of loyalty card and coupon match, but before manufacturer coupon reimbursement).
- Discounts (`RETAIL_DISC`, `COUPON_DISC`, `COUPON_MATCH_DISC`) are recorded as negative values or 0.
- Net customer out-of-pocket spend is: `SALES_VALUE + COUPON_DISC`.
- 14,466 records have `QUANTITY == 0` (mostly zero-dollar price adjustment entries or point of sale adjustments). Per guidelines, we preserve every original transaction record and flag these for selective filtering rather than discarding them.

In [3]:
# Clean transactions using modular cleaning pipeline
tx_clean = clean_transactions(tx_raw)

print(f"Cleaned transactions shape: {tx_clean.shape}")
print(f"Preserved exact original row count: {len(tx_clean) == len(tx_raw)}")
print(f"Zero quantity records flagged: {tx_clean['IS_ZERO_QUANTITY'].sum():,}")
print(f"Zero sales value records flagged: {tx_clean['IS_ZERO_SALES'].sum():,}")
print(f"Lines utilizing loyalty discounts: {tx_clean['HAS_RETAIL_DISC'].sum():,}")
print(f"Lines utilizing manufacturer coupons: {tx_clean['HAS_COUPON'].sum():,}")

display(tx_clean[[
    "household_key", "BASKET_ID", "DAY", "PRODUCT_ID", "QUANTITY", 
    "SALES_VALUE", "NET_CUSTOMER_SPEND", "RETAIL_DISC_AMOUNT", "COUPON_DISC_AMOUNT"
]].head())

Cleaned transactions shape: (2595732, 24)
Preserved exact original row count: True
Zero quantity records flagged: 14,466
Zero sales value records flagged: 18,850
Lines utilizing loyalty discounts: 1,303,026
Lines utilizing manufacturer coupons: 36,422


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,NET_CUSTOMER_SPEND,RETAIL_DISC_AMOUNT,COUPON_DISC_AMOUNT
0,2375,26984851472,1,1004906,1,1.39,1.39,0.60,-0.0
1,2375,26984851472,1,1033142,1,0.82,0.82,-0.00,-0.0
2,2375,26984851472,1,1036325,1,0.99,0.99,0.30,-0.0
3,2375,26984851472,1,1082185,1,1.21,1.21,-0.00,-0.0
4,2375,26984851472,1,8160430,1,1.50,1.50,0.39,-0.0


In [4]:
# Financial reconciliation check
total_sales_value = tx_clean["SALES_VALUE"].sum()
total_net_customer_spend = tx_clean["NET_CUSTOMER_SPEND"].sum()
total_retail_discounts = tx_clean["RETAIL_DISC_AMOUNT"].sum()
total_coupon_discounts = tx_clean["COUPON_DISC_AMOUNT"].sum()

print("Financial Summary across 2,595,732 transaction lines:")
print(f" - Total Retailer Sales Value:  ${total_sales_value:,.2f}")
print(f" - Total Customer Net Spend:    ${total_net_customer_spend:,.2f}")
print(f" - Total Retail Loyalty Disc:   ${total_retail_discounts:,.2f}")
print(f" - Total Mfr Coupon Disc:       ${total_coupon_discounts:,.2f}")
print(f" - Loyalty Savings as % of Sales: {(total_retail_discounts / total_sales_value) * 100:.2f}%")

Financial Summary across 2,595,732 transaction lines:
 - Total Retailer Sales Value:  $8,057,463.08
 - Total Customer Net Spend:    $8,014,851.54
 - Total Retail Loyalty Disc:   $1,398,334.84
 - Total Mfr Coupon Disc:       $42,611.54
 - Loyalty Savings as % of Sales: 17.35%


## 3. Product Catalog Cleaning
Cleaning whitespace across textual hierarchy (`DEPARTMENT`, `COMMODITY_DESC`, `SUB_COMMODITY_DESC`, `BRAND`) and converting empty size fields to `'UNKNOWN'`.

In [5]:
prod_clean = clean_products(prod_raw)

print(f"Cleaned products shape: {prod_clean.shape}")
print(f"Unique departments: {prod_clean['DEPARTMENT'].nunique()}")
print(f"Missing/Blank sizes standardized to UNKNOWN: {(prod_clean['CURR_SIZE_OF_PRODUCT'] == 'UNKNOWN').sum():,}")
print("Top 5 Departments by Product Count:")
display(prod_clean["DEPARTMENT"].value_counts().head(5))

Cleaned products shape: (92353, 7)
Unique departments: 44
Missing/Blank sizes standardized to UNKNOWN: 30,607
Top 5 Departments by Product Count:


DEPARTMENT
GROCERY      39021
DRUG GM      31529
PRODUCE       3118
COSMETICS     3011
NUTRITION     2914
Name: count, dtype: int64

## 4. Household Demographics Cleaning
Demographic information is available for 801 of the 2,500 households (32.0%). We clean whitespace and preserve the original categorical order.

In [6]:
demo_clean = clean_demographics(demo_raw)

print(f"Cleaned demographics shape: {demo_clean.shape}")
print(f"Unique households profiled: {demo_clean['household_key'].nunique()}")
print("Sample cleaned demographics:")
display(demo_clean.head())

Cleaned demographics shape: (801, 8)
Unique households profiled: 801
Sample cleaned demographics:


,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8
3,Age Group2,Y,Level6,Homeowner,Group4,4,2,13
4,Age Group4,Z,Level5,Homeowner,Group3,1,None/Unknown,16


## 5. Campaign & Coupon Data Cleaning & Deduplication
- `coupon.csv` contains 5,164 exact duplicate rows that must be removed.
- Calculate campaign duration in days.
- Verify relationship integrity for coupon redemptions.

In [7]:
camp_desc_clean, camp_table_clean = clean_campaigns(camp_desc_raw, camp_table_raw)
coupon_clean = clean_coupons(coupon_raw)
redempt_clean = clean_coupon_redemptions(redempt_raw)

print(f"Campaign descriptions cleaned: {len(camp_desc_clean)} campaigns")
print(f"Campaign targeting table cleaned: {len(camp_table_clean):,} rows ({camp_table_clean['household_key'].nunique()} unique households)")
print(f"Coupon catalog: {len(coupon_raw):,} raw -> {len(coupon_clean):,} cleaned (removed {len(coupon_raw) - len(coupon_clean):,} exact duplicates)")
print(f"Coupon redemptions cleaned: {len(redempt_clean):,} records ({redempt_clean['household_key'].nunique()} households)")

Campaign descriptions cleaned: 30 campaigns
Campaign targeting table cleaned: 7,208 rows (1584 unique households)
Coupon catalog: 124,548 raw -> 119,384 cleaned (removed 5,164 exact duplicates)
Coupon redemptions cleaned: 2,318 records (434 households)


## 6. Analytical Aggregations: Basket-Level Table (`baskets.csv`)
Aggregates product line transactions into individual basket shopping trips (Grain: `BASKET_ID`).

In [8]:
baskets = build_basket_summary(tx_clean)

print(f"Basket summary table shape: {baskets.shape}")
print(f"Unique baskets: {baskets['BASKET_ID'].nunique():,}")
print(f"Unique households represented: {baskets['household_key'].nunique():,}")
print(f"Average basket sales value: ${baskets['basket_sales'].mean():.2f}")
print(f"Average basket item count:  {baskets['line_item_count'].mean():.1f} items")
print(f"Baskets utilizing coupons:  {baskets['has_coupon_used'].sum():,} ({baskets['has_coupon_used'].mean()*100:.2f}%)")

display(baskets.head())

Basket summary table shape: (276484, 19)
Unique baskets: 276,484
Unique households represented: 2,500
Average basket sales value: $29.14
Average basket item count:  9.4 items
Baskets utilizing coupons:  16,751 (6.06%)


,BASKET_ID,household_key,DAY,WEEK_NO,STORE_ID,TRANS_TIME,line_item_count,distinct_product_count,total_quantity,basket_sales,basket_net_spend,retail_disc_amount,coupon_disc_amount,coupon_match_amount,total_discount_amount,has_coupon_used,has_retail_disc_used,avg_line_item_value,discount_rate_pct
0,26984851472,2375,1,1,364,1631,5,5,5,5.91,5.91,1.29,-0.0,-0.0,1.29,False,True,1.18,17.9
1,26984851516,2375,1,1,364,1642,6,6,7,12.43,12.43,3.26,-0.0,-0.0,3.26,False,True,2.07,20.8
2,26984896261,1364,1,1,31742,1520,5,5,5,11.37,11.37,2.18,-0.0,-0.0,2.18,False,True,2.27,16.1
3,26984905972,1130,1,1,31642,1340,5,5,9,2.55,2.55,2.08,-0.0,-0.0,2.08,False,True,0.51,44.9
4,26984945254,1173,1,1,412,2042,3,3,4,3.53,3.53,-0.00,-0.0,-0.0,-0.00,False,False,1.18,-0.0


## 7. Analytical Aggregations: Household-Level Table (`household_summary.csv`)
Summarizes overall 2-year customer metrics (Grain: `household_key`, 2,500 households).
Attaches demographic attributes where available, marking unprofiled households as `'Unprofiled'`.

In [9]:
household_summary = build_household_summary(tx_clean, baskets, demo_clean)

print(f"Household summary shape: {household_summary.shape}")
print(f"Total households: {len(household_summary):,}")
print(f"Profiled households: {household_summary['has_demographics'].sum():,}")
print(f"Mean household spend: ${household_summary['total_sales'].mean():,.2f}")
print(f"Median household spend: ${household_summary['total_sales'].median():,.2f}")
print(f"Mean basket count: {household_summary['basket_count'].mean():.1f} trips")

display(household_summary.head())

Household summary shape: (2500, 26)
Total households: 2,500
Profiled households: 801
Mean household spend: $3,222.99
Median household spend: $2,157.75
Mean basket count: 110.6 trips


,household_key,basket_count,total_sales,total_net_spend,total_quantity,total_discounts,retail_discounts,coupon_discounts,baskets_with_coupons,first_purchase_day,...,avg_basket_units,coupon_trip_ratio,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,has_demographics
0,1,86,4330.16,4249.60,1997,803.75,697.04,80.56,29,51,...,23.22,0.337,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,True
1,2,45,1954.34,1945.34,834,343.99,334.99,9.00,5,103,...,18.53,0.111,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,False
2,3,47,2653.21,2592.02,8540,756.15,675.16,61.19,5,113,...,181.70,0.106,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,False
3,4,30,1200.11,1197.61,382,118.15,115.65,2.50,1,104,...,12.73,0.033,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,False
4,5,40,779.06,779.06,245,118.33,118.33,0.00,0,85,...,6.12,0.000,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,Unprofiled,False


## 8. Export Cleaned Datasets to `data/processed/`
Exporting all cleaned datasets in standard CSV formats (and Parquet for the large transaction table) to serve downstream notebooks, SQL scripts, and Power BI dashboards.

In [10]:
processed_dir = ROOT_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save transactions as both parquet and csv
print("Saving cleaned transactions...")
tx_clean.to_parquet(processed_dir / "cleaned_transactions.parquet", index=False)
tx_clean.to_csv(processed_dir / "cleaned_transactions.csv", index=False)

# Save remaining tables as CSV
print("Saving cleaned dimensions and analytical summaries...")
prod_clean.to_csv(processed_dir / "cleaned_products.csv", index=False)
demo_clean.to_csv(processed_dir / "cleaned_demographics.csv", index=False)
camp_desc_clean.to_csv(processed_dir / "cleaned_campaigns.csv", index=False)
camp_table_clean.to_csv(processed_dir / "cleaned_campaign_households.csv", index=False)
coupon_clean.to_csv(processed_dir / "cleaned_coupons.csv", index=False)
redempt_clean.to_csv(processed_dir / "cleaned_coupon_redemptions.csv", index=False)
baskets.to_csv(processed_dir / "baskets.csv", index=False)
household_summary.to_csv(processed_dir / "household_summary.csv", index=False)

print("All 9 processed datasets saved successfully in data/processed/!")

Saving cleaned transactions...


Saving cleaned dimensions and analytical summaries...


All 9 processed datasets saved successfully in data/processed/!


## 9. Export Verification & Manifest
Validating file sizes and integrity of exported data artifacts.

In [11]:
files = list(processed_dir.glob("*"))
manifest = []

for f in sorted(files):
    size_mb = f.stat().st_size / (1024 * 1024)
    manifest.append({
        "Filename": f.name,
        "Size (MB)": round(size_mb, 2),
        "Format": f.suffix
    })

manifest_df = pd.DataFrame(manifest)
display(manifest_df)
print("Data cleaning stage completed successfully!")

,Filename,Size (MB),Format
0,baskets.csv,24.81,.csv
1,cleaned_campaign_households.csv,0.09,.csv
2,cleaned_campaigns.csv,0.00,.csv
3,cleaned_coupon_redemptions.csv,0.05,.csv
4,cleaned_coupons.csv,2.58,.csv
5,cleaned_demographics.csv,0.04,.csv
6,cleaned_products.csv,6.31,.csv
7,cleaned_transactions.csv,303.11,.csv
8,cleaned_transactions.parquet,40.31,.parquet
9,household_summary.csv,0.38,.csv


Data cleaning stage completed successfully!
